# Catalog extensions

An extension holds additional columns for a HATS catalog. It can be stored in a separate location, and is only read when requested.

This notebook runs the whole pipeline that we have so far:

1. **hats-import** splits a catalog into a "core" and an extension, written as one collection
2. **hats** reads the collection, the extension's properties and the extension data
3. **LSDB** opens the core and joins the extension back to it, with `load_extension`
4. Demonstrate that the extension can be stored and loaded from outside our collection.

In [1]:
import hats
import logging
import lsdb
import shutil

import pandas as pd
from dask.distributed import Client
from hats.catalog import ExtensionProperties, CollectionProperties
from hats_import import ExtensionArguments, pipeline_with_client
from upath import UPath

logging.getLogger().setLevel(logging.ERROR)
logging.getLogger("distributed").setLevel(logging.WARNING)

client = Client(n_workers=1, threads_per_worker=1, local_directory="/tmp/dask")

## 1. Split an existing Catalog with `hats-import`

The input is the `small_sky_order1_nested_sources` collection: 131 objects, each with a nested `sources` column holding its light curve, plus a margin and two indexes.

In [2]:
input_collection = hats.read_hats("../tests/data/small_sky_order1_nested_sources")
print("columns:", input_collection.main_catalog.schema.names)
print("margins:", input_collection.all_margins)
print("indexes:", input_collection.all_indexes)

columns: ['id', 'ra', 'dec', 'ra_error', 'dec_error', 'sources', '_healpix_29']
margins: ['small_sky_order1_nested_sources_margin']
indexes: {'id': 'small_sky_order1_object_id_index', 'sources.source_id': 'small_sky_order1_source_id_index'}


- `ExtensionArguments` defines `sources` as the column to move into an extension named `lightcurves`.
- The extension joins back to the core on `id`, stored as `object_id` in the extension, and keeps the partitioning of the input.
- The margin is split too, so the core and the extension each get one.

In [3]:
args = ExtensionArguments(
    input_catalog_path="../tests/data/small_sky_order1_nested_sources",
    extension_columns=["sources"],
    primary_column="id",
    join_column="object_id",
    extension_alias="lightcurves",
    extension_name="small_sky_order1_lightcurves",
    product_type_served="lightcurve",
    output_path=".",
    output_artifact_name="small_sky_order1_with_extension",
    progress_bar=False,
)
pipeline_with_client(args, client)

In [4]:
!tree -L 2 "small_sky_order1_with_extension"

small_sky_order1_with_extension
├── collection.properties
├── small_sky_order1_lightcurves
│   ├── collection.properties
│   ├── small_sky_order1_lightcurves
│   ├── small_sky_order1_lightcurves_margin
│   └── small_sky_order1_source_id_index
├── small_sky_order1_lightcurves.properties
├── small_sky_order1_nested_sources
│   ├── dataset
│   ├── hats.properties
│   ├── partition_info.csv
│   ├── point_map.fits
│   ├── properties
│   └── skymap.fits
├── small_sky_order1_nested_sources_margin
│   ├── dataset
│   ├── hats.properties
│   ├── partition_info.csv
│   └── properties
└── small_sky_order1_object_id_index
    ├── dataset
    ├── hats.properties
    └── properties

11 directories, 13 files


In [5]:
%cat "small_sky_order1_with_extension/collection.properties"

#HATS Collection
obs_collection=small_sky_order1_with_extension
hats_primary_table_url=small_sky_order1_nested_sources
all_margins=small_sky_order1_nested_sources_margin
default_margin=small_sky_order1_nested_sources_margin
all_indexes=id small_sky_order1_object_id_index
all_extensions=lightcurves small_sky_order1_lightcurves.properties
hats_builder=hats-import v0.11.1.dev12+g033d09f92, hats v0.11.1.dev23+g711c5ab47
hats_creation_date=2026-10-15T16:26UTC
hats_estsize=867
hats_release_date=2025-08-22
hats_version=v1.0


## 2. Read the extension info with hats

The output is a collection. Its `collection.properties` lists the extension in `all_extensions`, next to the margins and indexes.

In [6]:
collection = hats.read_hats("small_sky_order1_with_extension")
print(type(collection))
print("main:", collection.main_catalog_dir)
print("margins:", collection.all_margins)
print("indexes:", collection.all_indexes)
print("extensions:", collection.all_extensions)

<class 'hats.catalog.catalog_collection.CatalogCollection'>
main: small_sky_order1_with_extension/small_sky_order1_nested_sources
margins: ['small_sky_order1_nested_sources_margin']
indexes: {'id': 'small_sky_order1_object_id_index'}
extensions: {'lightcurves': 'small_sky_order1_lightcurves.properties'}


The main table and main margin columns are now stripped of the extension columns:

In [7]:
print("core columns:", collection.main_catalog.schema.names)

core columns: ['id', 'ra', 'dec', 'ra_error', 'dec_error', '_healpix_29']


The extension is described by its `<extension>.properties` file: where its data is, the columns to join on, the columns it provides, the join style and its modality. The pipeline creates it at the root of the collection.

In [8]:
properties_path = "small_sky_order1_with_extension/small_sky_order1_lightcurves.properties"
!cat $properties_path

#HATS Extension
obs_collection=small_sky_order1_lightcurves
dataproduct_type=extension
hats_primary_table_url=small_sky_order1_with_extension
hats_col_assn_primary=id
hats_assn_join_table_url=small_sky_order1_lightcurves
hats_col_assn_join=object_id
hats_ext_cols=sources
hats_ext_join_style=left
hats_ext_shares_primary_coords=True
hats_product_type_served=lightcurve
hats_builder=hats-import v0.11.1.dev12+g033d09f92, hats v0.11.1.dev23+g711c5ab47
hats_creation_date=2026-10-15T16:26UTC
hats_estsize=804
hats_release_date=2025-08-22
hats_version=v1.0


`hats.read_hats` on that file returns a `CatalogExtension`, with its properties and its data: a catalog, or a collection when it has margins, as here.

In [9]:
hc_extension = hats.read_hats(properties_path)
print(type(hc_extension))
print("path to the data:", hc_extension.join_catalog_dir)
print("extension columns:", hc_extension.catalog.main_catalog.schema.names)

<class 'hats.catalog.catalog_extension.CatalogExtension'>
path to the data: small_sky_order1_with_extension/small_sky_order1_lightcurves
extension columns: ['_healpix_29', 'object_id', 'ra', 'dec', 'sources']


**Note:** The pipeline copied the healpix column and the ra/dec to the extension.

## 3. Load the extension data with LSDB


In [10]:
lsdb.catalog.Catalog.load_extension?

Signature:
lsdb.catalog.Catalog.load_extension(
    self,
    name_or_path: 'str | Path | UPath',
    columns: 'str | list[str] | None' = None,
    storage_options: 'dict | None' = None,
) -> 'Catalog'
Docstring:
Load an extension of the catalog.

An extension holds additional columns for the rows of a catalog, stored apart from it.
It is described by an ``<extension>.properties`` file, which defines the columns to join on.
An extension listed in the catalog's collection (see `all_extensions`) can be loaded by its
name; any other extension is loaded by the path to its ``<extension>.properties`` file.

The extension is joined with `Catalog.join`, in the join style that the extension specifies:
rows of the catalog with no extension data are kept, with missing values, when it is "left"
(the default), and dropped when it is "inner".

The loaded extensions are listed in `extensions`.

Parameters
----------
name_or_path : str or path-like
    The name of an extension in the catalog's collect

### 3.1. Open the "core" collection

Opening the collection loads the "core" columns only. `all_extensions` lists the extensions that can be loaded, and `extensions` the ones that are. We decided on this terminology to mirror `columns`/`all_columns`.

In [11]:
objects = lsdb.open_catalog("small_sky_order1_with_extension")
print("all_extensions:", objects.all_extensions)
print("extensions:", objects.extensions)
objects

all_extensions: {'lightcurves': 'small_sky_order1_lightcurves.properties'}
extensions: ()


,id,ra,dec,ra_error,dec_error
npartitions=4,,,,,
"Order: 1, Pixel: 44",int64[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],int64[pyarrow]
"Order: 1, Pixel: 45",...,...,...,...,...
"Order: 1, Pixel: 46",...,...,...,...,...
"Order: 1, Pixel: 47",...,...,...,...,...


### 3.2. Join the extension to the objects

`load_extension` reads the extension columns and joins them to the core, on the columns and with the join style that the extension properties specify. The extension's own join column is dropped.

In [12]:
extended = objects.load_extension("lightcurves")
print("extensions:", extended.extensions)
extended

extensions: ('lightcurves',)


,id,ra,dec,ra_error,dec_error,sources
npartitions=4,,,,,,
"Order: 1, Pixel: 44",int64[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],int64[pyarrow],"nested<source_id: [int64], source_ra: [double]..."
"Order: 1, Pixel: 45",...,...,...,...,...,...
"Order: 1, Pixel: 46",...,...,...,...,...,...
"Order: 1, Pixel: 47",...,...,...,...,...,...


In [13]:
extended.head()

Building task graph...


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

id     ra   dec  ra_error  dec_error  \
_healpix_29                                                  
3187422220182231470  707  308.5 -69.5         0          0   
3187796123455121090  792  320.5 -69.5         0          0   
3188300701662669945  723  315.5 -68.5         0          0   
3188300701662669945  811  315.5 -68.5         0          0   
3192670279995812269  826  335.5 -69.5         0          0   

                                                               sources  
_healpix_29                                                             
3187422220182231470  [{source_id: 74859, source_ra: 308.584555, sou...  
3187796123455121090  [{source_id: 79856, source_ra: 320.518757, sou...  
3188300701662669945  [{source_id: 82780, source_ra: 315.768603, sou...  
3188300701662669945  [{source_id: 81352, source_ra: 316.043222, sou...  
3192670279995812269  [{source_id: 71849, source_ra: 335.877695, sou...

The extended catalog has the same rows and columns as the catalog we started from:

In [14]:
original = lsdb.open_catalog("../tests/data/small_sky_order1_nested_sources").compute()
result = extended.compute()
pd.testing.assert_frame_equal(result, original[result.columns])

Building task graph...


Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

Building task graph...


Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

### 3.3. Load part of the extension columns

We can load only part of the extension columns. `columns` can name nested subcolumns, and only those are read from disk:

In [15]:
objects.load_extension("lightcurves", columns=["sources.mjd", "sources.mag"]).head()

Building task graph...


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

id     ra   dec  ra_error  dec_error  \
_healpix_29                                                  
3187422220182231470  707  308.5 -69.5         0          0   
3187796123455121090  792  320.5 -69.5         0          0   
3188300701662669945  723  315.5 -68.5         0          0   
3188300701662669945  811  315.5 -68.5         0          0   
3192670279995812269  826  335.5 -69.5         0          0   

                                                               sources  
_healpix_29                                                             
3187422220182231470  [{mjd: 58702.504576, mag: 16.338498}; …] (131 ...  
3187796123455121090  [{mjd: 59049.869123, mag: 18.560216}; …] (131 ...  
3188300701662669945  [{mjd: 59258.882205, mag: 19.624363}; …] (131 ...  
3188300701662669945  [{mjd: 59154.857324, mag: 20.855713}; …] (131 ...  
3192670279995812269  [{mjd: 58490.377595, mag: 16.24181}; …] (131 r...

### 3.4. Load an extension as a standalone catalog

An Extension is a valid catalog and can be read as a standalone Catalog. `lsdb.read_hats` on the extension's `.properties` file reads its data, with its margin.

In [16]:
extension_data = lsdb.read_hats("small_sky_order1_with_extension/small_sky_order1_lightcurves")
print("margin:", extension_data.margin.name, extension_data.margin.hc_structure.catalog_info.margin_threshold)
extension_data

margin: small_sky_order1_lightcurves_margin 7200.0


,object_id,ra,dec,sources
npartitions=4,,,,
"Order: 1, Pixel: 44",int64[pyarrow],double[pyarrow],double[pyarrow],"nested<source_id: [int64], source_ra: [double]..."
"Order: 1, Pixel: 45",...,...,...,...
"Order: 1, Pixel: 46",...,...,...,...
"Order: 1, Pixel: 47",...,...,...,...


### 3.5. Load an extension from its path

An extension does not need to be listed in a collection: it can also be loaded from the path to its `.properties` file, under any name. Here, onto the core catalog opened on its own, outside of the collection.

In [17]:
core = lsdb.open_catalog("small_sky_order1_with_extension/small_sky_order1_nested_sources")
print("all_extensions:", core.all_extensions)
from_path = core.load_extension("small_sky_order1_with_extension/small_sky_order1_lightcurves.properties")
print("extensions:", from_path.extensions)
pd.testing.assert_frame_equal(from_path.compute(), result)

Building task graph...


all_extensions: {}
extensions: ('small_sky_order1_lightcurves',)


Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

## 4. Extensions stored outside of the collection

An extension can live anywhere:
- The path to `hats_assn_join_table_url` in the properties file may be absolute.
- The path to the properties file itself may be absolute, when listed in `collection.properties` or when passed to `load_extension`.

### 4.1 The extension data elsewhere

The `.properties` file stays in the collection, and points to the extension data by absolute path.

In [18]:
collection_dir = UPath("small_sky_order1_with_extension")
shutil.copytree(collection_dir / "small_sky_order1_lightcurves", "copy_of_extension", dirs_exist_ok=True)
properties = ExtensionProperties.read_from_file(collection_dir / "small_sky_order1_lightcurves.properties")
properties.model_copy(update={"join_catalog": "../copy_of_extension"}).to_properties_file(collection_dir)

In [19]:
!cat $collection_dir/small_sky_order1_lightcurves.properties

#HATS Extension
obs_collection=small_sky_order1_lightcurves
dataproduct_type=extension
hats_primary_table_url=small_sky_order1_with_extension
hats_col_assn_primary=id
hats_assn_join_table_url=../copy_of_extension
hats_col_assn_join=object_id
hats_ext_cols=sources
hats_ext_join_style=left
hats_ext_shares_primary_coords=True
hats_product_type_served=lightcurve
hats_builder=hats-import v0.11.1.dev12+g033d09f92, hats v0.11.1.dev23+g711c5ab47
hats_creation_date=2026-10-15T16:26UTC
hats_estsize=804
hats_release_date=2025-08-22
hats_version=v1.0


In [20]:
lsdb.open_catalog(collection_dir).load_extension("lightcurves").head()

Building task graph...


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

id     ra   dec  ra_error  dec_error  \
_healpix_29                                                  
3187422220182231470  707  308.5 -69.5         0          0   
3187796123455121090  792  320.5 -69.5         0          0   
3188300701662669945  723  315.5 -68.5         0          0   
3188300701662669945  811  315.5 -68.5         0          0   
3192670279995812269  826  335.5 -69.5         0          0   

                                                               sources  
_healpix_29                                                             
3187422220182231470  [{source_id: 74859, source_ra: 308.584555, sou...  
3187796123455121090  [{source_id: 79856, source_ra: 320.518757, sou...  
3188300701662669945  [{source_id: 82780, source_ra: 315.768603, sou...  
3188300701662669945  [{source_id: 81352, source_ra: 316.043222, sou...  
3192670279995812269  [{source_id: 71849, source_ra: 335.877695, sou...

### 4.2 The `.properties` file elsewhere

The extension, with its `.properties` file, is moved out of the collection.

In [ ]:
properties.model_copy(update={"join_catalog": "copy_of_extension"}).to_properties_file(".")
properties = CollectionProperties.read_from_dir(collection_dir)
properties.model_copy(update={"all_extensions": {"lightcurves": "../small_sky_order1_lightcurves.properties"}}).to_properties_file(collection_dir)

In [22]:
!cat $collection_dir/collection.properties

#HATS Collection
obs_collection=small_sky_order1_with_extension
hats_primary_table_url=small_sky_order1_nested_sources
all_margins=small_sky_order1_nested_sources_margin
default_margin=small_sky_order1_nested_sources_margin
all_indexes=id small_sky_order1_object_id_index
all_extensions=lightcurves ../small_sky_order1_lightcurves.properties
hats_builder=hats-import v0.11.1.dev12+g033d09f92, hats v0.11.1.dev23+g711c5ab47
hats_creation_date=2026-10-15T16:26UTC
hats_estsize=867
hats_release_date=2025-08-22
hats_version=v1.0


In [23]:
lsdb.open_catalog(collection_dir).load_extension("lightcurves").head()

Building task graph...


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

id     ra   dec  ra_error  dec_error  \
_healpix_29                                                  
3187422220182231470  707  308.5 -69.5         0          0   
3187796123455121090  792  320.5 -69.5         0          0   
3188300701662669945  723  315.5 -68.5         0          0   
3188300701662669945  811  315.5 -68.5         0          0   
3192670279995812269  826  335.5 -69.5         0          0   

                                                               sources  
_healpix_29                                                             
3187422220182231470  [{source_id: 74859, source_ra: 308.584555, sou...  
3187796123455121090  [{source_id: 79856, source_ra: 320.518757, sou...  
3188300701662669945  [{source_id: 82780, source_ra: 315.768603, sou...  
3188300701662669945  [{source_id: 81352, source_ra: 316.043222, sou...  
3192670279995812269  [{source_id: 71849, source_ra: 335.877695, sou...

In [24]:
client.close()